# LSTM Catch22

### Imports y cargas de CSV

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from keras.models import Model
from keras.layers import LSTM, Dense, Dropout, Input, Concatenate
from keras.callbacks import EarlyStopping
from keras.optimizers import Adam
from sklearn.metrics import mean_squared_error, mean_absolute_error

tf.random.set_seed(123)
np.random.seed(123)

VENTANA = 6  # misma ventana del Modelo A ganador, para que la comparacion sea justa

/Users/marinesgarcia/Documents/UVG/DS/Lab1DS/lab1ds/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [6]:
def cargar_serie(path):
    s = pd.read_csv(path, index_col=0, parse_dates=True)['Viajero']
    s = s.asfreq('MS')
    return s

train_serie = cargar_serie('../../Datos/train_la_aurora.csv')
test_serie  = cargar_serie('../../Datos/test_la_aurora.csv')

matriz_std = pd.read_csv('../../catch22_features/matriz_std.csv', index_col=0)

print(f"Train: {len(train_serie)} obs | Test: {len(test_serie)} obs")
print(f"Series en matriz_std: {matriz_std.index.tolist()}")

Train: 147 obs | Test: 63 obs
Series en matriz_std: ['Marítima', 'Valle Nuevo', 'Total', 'San Cristobal', 'Vía Terrestre', 'Vía Aérea', 'La Aurora']


### Vector catch22 de La Aurora

In [7]:

# Vector catch22 de La Aurora — usar el YA ESTANDARIZADO de tu matriz de equipo
# (el mismo matriz_std que armaste en 2.4), NO recalcularlo de nuevo aqui.

catch22_vector = matriz_std.loc['La Aurora'].values.astype(float)  # shape (22,)

def crear_secuencias(datos_escalados, ventana):
    X, y = [], []
    for i in range(len(datos_escalados) - ventana):
        X.append(datos_escalados[i:i+ventana, 0])
        y.append(datos_escalados[i+ventana, 0])
    return np.array(X), np.array(y)

valores = train_serie.values.reshape(-1, 1).astype(float)
scaler = MinMaxScaler(feature_range=(0, 1))
valores_esc = scaler.fit_transform(valores)

X, y = crear_secuencias(valores_esc, VENTANA)
X = X.reshape(X.shape[0], X.shape[1], 1)
X_catch22 = np.tile(catch22_vector, (X.shape[0], 1))  # el mismo vector repetido para cada ventana de esta serie

corte = int(len(X) * 0.85)
X_train, y_train = X[:corte], y[:corte]
X_val, y_val = X[corte:], y[corte:]
Xc_train, Xc_val = X_catch22[:corte], X_catch22[corte:]

### Arquitectura: rama LSTM (secuencia) + rama densa (catch22) -> se combina

In [8]:

entrada_secuencia = Input(shape=(VENTANA, 1), name='secuencia')
rama_lstm = LSTM(32)(entrada_secuencia)
rama_lstm = Dropout(0.1)(rama_lstm)

entrada_catch22 = Input(shape=(22,), name='catch22')
rama_estatica = Dense(16, activation='relu')(entrada_catch22)

combinado = Concatenate()([rama_lstm, rama_estatica])
salida = Dense(1)(combinado)

modelo_catch22 = Model(inputs=[entrada_secuencia, entrada_catch22], outputs=salida)
modelo_catch22.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

es = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
modelo_catch22.fit([X_train, Xc_train], y_train,
                    validation_data=([X_val, Xc_val], y_val),
                    epochs=150, batch_size=8, callbacks=[es], verbose=0)

### Pronostico recursivo (el vector catch22 se mantiene fijo en cada paso,solo la ventana de la secuencia se va actualizando)

In [9]:
ultima_ventana = valores_esc[-VENTANA:].reshape(1, VENTANA, 1)
catch22_input = catch22_vector.reshape(1, -1)
predicciones_esc = []
ventana_actual = ultima_ventana.copy()
for _ in range(len(test_serie)):
    pred = modelo_catch22.predict([ventana_actual, catch22_input], verbose=0)[0, 0]
    predicciones_esc.append(pred)
    ventana_actual = np.append(ventana_actual[:, 1:, :], [[[pred]]], axis=1)

predicciones_catch22 = scaler.inverse_transform(np.array(predicciones_esc).reshape(-1, 1)).flatten()
predicciones_catch22 = pd.Series(predicciones_catch22, index=test_serie.index)

mae_catch22 = mean_absolute_error(test_serie, predicciones_catch22)
rmse_catch22 = np.sqrt(mean_squared_error(test_serie, predicciones_catch22))

### Comparación Final

In [10]:
comparacion = pd.DataFrame({
    'Modelo': ['LSTM + catch22', 'Mejor LSTM anterior (Modelo A, ventana=6)', 'Prophet (Lab1)'],
    'MAE':  [mae_catch22, 18321.89, 45971.28],
    'RMSE': [rmse_catch22, 23376.72, 50960.97],
}).sort_values('RMSE')
print(comparacion.to_string(index=False))

                                   Modelo          MAE         RMSE
Mejor LSTM anterior (Modelo A, ventana=6) 18321.890000 23376.720000
                           LSTM + catch22 23086.722263 28703.839233
                           Prophet (Lab1) 45971.280000 50960.970000
